# In this Notebook we attempt to find the best model
We compare the logistic regression, decision Trees and random forest model to find the best model for predicition of occupancy.
- we later also tune the models and then compare the performance of the models after tuning them to get the best performance

In [ ]:
# ===============================================
# BASIC SETUP: GIT Auth and mounting google drive
# ================================================

from google.colab import drive, userdata
import sys

# Standard mount to access the config file initially
drive.mount('/content/drive')
PROJECT_ROOT = userdata.get('BIG_DATA_PATH')
sys.path.append(PROJECT_ROOT + "/src/")

from utils.config import initialize_project
initialize_project()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
train_df = pd.read_csv("data/raw/datatraining.txt")
test1_df = pd.read_csv("data/raw/datatest.txt")
test2_df = pd.read_csv("data/raw/datatest2.txt")

In [ ]:
print("Train shape:", train_df.shape)
print("Test1 shape:", test1_df.shape)
print("Test2 shape:", test2_df.shape)

In [ ]:
features = ["Temperature", "Humidity", "Light", "CO2", "HumidityRatio"]
target = "Occupancy"

X_train = train_df[features]
y_train = train_df[target]

X_test1 = test1_df[features]
y_test1 = test1_df[target]

X_test2 = test2_df[features]
y_test2 = test2_df[target]

In [ ]:
# evaluation function
def evaluate_model(model_name, y_true, y_pred):
    print(f"--- {model_name} ---")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred), 4))
    print("Recall:", round(recall_score(y_true, y_pred), 4))
    print("F1 Score:", round(f1_score(y_true, y_pred), 4))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
# scaling the logistic regression features
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)

In [ ]:
lr_pred_test1 = lr_pipeline.predict(X_test1)
lr_pred_test2 = lr_pipeline.predict(X_test2)

evaluate_model("Logistic Regression - Test1", y_test1, lr_pred_test1)
evaluate_model("Logistic Regression - Test2", y_test2, lr_pred_test2)

In [ ]:
# Random Forest Model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

In [ ]:
rf_pred_test1 = rf_model.predict(X_test1)
rf_pred_test2 = rf_model.predict(X_test2)

evaluate_model("Random Forest - Test1", y_test1, rf_pred_test1)
evaluate_model("Random Forest - Test2", y_test2, rf_pred_test2)

In [ ]:
# Basic Decision Tree Model
dt_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

dt_model.fit(X_train, y_train)

In [ ]:
dt_pred_test1 = dt_model.predict(X_test1)
dt_pred_test2 = dt_model.predict(X_test2)

evaluate_model("Decision Tree - Test1", y_test1, dt_pred_test1)
evaluate_model("Decision Tree - Test2", y_test2, dt_pred_test2)

In [ ]:
# model comparison table
results = []

def add_result(model_name, dataset_name, y_true, y_pred):
    results.append({
        "Model": model_name,
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1 Score": f1_score(y_true, y_pred)
    })

add_result("Logistic Regression", "Test1", y_test1, lr_pred_test1)
add_result("Logistic Regression", "Test2", y_test2, lr_pred_test2)

add_result("Decision Tree", "Test1", y_test1, dt_pred_test1)
add_result("Decision Tree", "Test2", y_test2, dt_pred_test2)

add_result("Random Forest", "Test1", y_test1, rf_pred_test1)
add_result("Random Forest", "Test2", y_test2, rf_pred_test2)

results_df = pd.DataFrame(results)
results_df

In [ ]:
# sort values
results_df.sort_values(by=["Dataset", "F1 Score"], ascending=False)

In [ ]:
# plot model comparison
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="Model", y="F1 Score", hue="Dataset")
plt.title("Model Comparison by F1 Score")
plt.ylim(0, 1)
plt.show()

In [ ]:
# Logistic regression coefficients
lr_model = lr_pipeline.named_steps["model"]

coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": lr_model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

coef_df

In [ ]:
# Random forest coefficients
rf_importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

rf_importance_df

In [ ]:
# We are trying to find the best hyperparameters that produce the highest results for the Random forest model
depths = [3, 5, 10, None]

for d in depths:
    rf = RandomForestClassifier(n_estimators=100, max_depth=d, random_state=42)
    rf.fit(X_train, y_train)

    pred = rf.predict(X_test2)
    print(f"Depth {d} → F1:", f1_score(y_test2, pred))

In [ ]:
# we are trying to find the best hyperparameters for the logsitic regression model by using regularization
# this will produce the best tuned logistic regression model
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])
param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__penalty": ["l2"]  # keep it simple for now
}
grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1
)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)

best_lr_model = grid.best_estimator_
lr_pred_test1 = best_lr_model.predict(X_test1)
lr_pred_test2 = best_lr_model.predict(X_test2)
evaluate_model("Tuned Logistic Regression - Test1", y_test1, lr_pred_test1)
evaluate_model("Tuned Logistic Regression - Test2", y_test2, lr_pred_test2)

After hyperparameter tuning, both Logistic Regression and Random Forest achieved near-identical performance. This indicates that the underlying relationship between environmental features and occupancy is predominantly linear, with limited nonlinear complexity.


# Final Takeaway
The optimal regularization strength suggests that a simpler linear model generalizes best, reinforcing that the signal in the data is strong and not overly complex.